In [1]:
import os
import pandas as pd
import numpy as np
from IPython.display import display

# Mendapatkan direktori aktif notebook
script_dir = os.getcwd() 

# List subjek dari SUB1 sampai SUB12
subjects = [f"SUB{i}" for i in range(1, 13)]

# Struktur baris data untuk tabel
data_rows = []

print("Mengekstrak dan menghitung nilai CER dari seluruh eksperimen...\n")

# ============================================================================
# 1. Eksperimen Single Subject (File Terpisah per Subjek)
# ============================================================================
single_experiments = [
    {
        'Decoder': 'LSTM',
        'Train Data': 'Single',
        'template': "{subject}_eq_3_0_fixed_hilbert_test_predictions_6_1.csv"
    },
    {
        'Decoder': 'IndoGPT',
        'Train Data': 'Single',
        'template': "{subject}_eq_3_0_hilbert_test_predictions_10_1_IndoGPT.csv"
    }
]

for exp in single_experiments:
    row_data = {
        'Decoder': exp['Decoder'],
        'Train Data': exp['Train Data']
    }
    
    # Ambil nilai per subjek dari file terpisah
    for subject in subjects:
        filename = exp['template'].format(subject=subject)
        filepath = os.path.join(script_dir, filename)
        
        if os.path.exists(filepath):
            try:
                df = pd.read_csv(filepath)
                row_data[subject] = round(df['cer'].mean(), 4)
            except Exception as e:
                print(f"  [ERROR] Gagal membaca {filename}: {e}")
                row_data[subject] = np.nan
        else:
            row_data[subject] = np.nan
            
    # Hitung rata-rata akhir untuk row ini
    valid_values = [row_data[sub] for sub in subjects if pd.notna(row_data[sub])]
    row_data['Rata-rata Global'] = round(np.mean(valid_values), 4) if valid_values else np.nan
    data_rows.append(row_data)

# ============================================================================
# 2. Eksperimen Cross-Subject / All Subjects (Satu File Gabungan)
# ============================================================================
all_experiments = [
    {
        'Decoder': 'LSTM',
        'Train Data': 'All',
        'filename': "all_eq_3_0_fixed_hilbert_test_predictions_1_1.csv"
    },
    {
        'Decoder': 'IndoGPT',
        'Train Data': 'All',
        'filename': "all_eq_3_0_hilbert_test_predictions_2_1_IndoGPT.csv"
    }
]

for exp in all_experiments:
    row_data = {
        'Decoder': exp['Decoder'],
        'Train Data': exp['Train Data']
    }
    
    filepath = os.path.join(script_dir, exp['filename'])
    
    if os.path.exists(filepath):
        try:
            # Muat file gabungan sekali saja
            df_all = pd.read_csv(filepath)
            
            # Filter internal berdasarkan kolom 'subject'
            for subject in subjects:
                sub_df = df_all[df_all['subject'] == subject]
                if len(sub_df) > 0:
                    row_data[subject] = round(sub_df['cer'].mean(), 4)
                else:
                    row_data[subject] = np.nan
        except Exception as e:
            print(f"  [ERROR] Gagal membaca {exp['filename']}: {e}")
            for subject in subjects:
                row_data[subject] = np.nan
    else:
        print(f"  ⚠️ [INFO] File {exp['filename']} belum ditemukan.")
        for subject in subjects:
            row_data[subject] = np.nan

    # Hitung rata-rata akhir untuk row ini
    valid_values = [row_data[sub] for sub in subjects if pd.notna(row_data[sub])]
    row_data['Rata-rata Global'] = round(np.mean(valid_values), 4) if valid_values else np.nan
    data_rows.append(row_data)

# ============================================================================
# MEMBUAT DAN MENAMPILKAN TABEL AKHIR
# ============================================================================
df_results = pd.DataFrame(data_rows)

# Atur urutan kolom agar 'Rata-rata Global' selalu berada di ujung kanan
column_order = ['Decoder', 'Train Data'] + subjects + ['Rata-rata Global']
df_results = df_results[column_order]

print("=" * 100)
print("TABEL PERBANDINGAN PERFORMA CER EKSPERIMEN HILBERT")
print("=" * 100)

display(df_results)

# Tabel OOV


# ==========================================
# KONFIGURASI PATH & PARAMETER
# ==========================================
CURRENT_DIR = os.getcwd()
DATASET_DIR = os.path.abspath(os.path.join(CURRENT_DIR, '../../../dataset'))

# Buat daftar subjek SUB1 sampai SUB12
subjects = [f'SUB{i}' for i in range(1, 13)]

# List untuk menyimpan hasil OOV agar bisa dijadikan tabel
oov_summary_list = []

for subject in subjects:
    # Definisikan nama file yang sudah displit
    train_file = os.path.join(DATASET_DIR, f"{subject}_eq_3_0_train.csv")
    val_file = os.path.join(DATASET_DIR, f"{subject}_eq_3_0_val.csv")
    test_file = os.path.join(DATASET_DIR, f"{subject}_eq_3_0_test.csv")
    
    # Periksa apakah ketiga file untuk subjek ini benar-benar ada
    if os.path.exists(train_file) and os.path.exists(val_file) and os.path.exists(test_file):
        # 1. Load Data
        df_train = pd.read_csv(train_file)
        df_val = pd.read_csv(val_file)
        df_test = pd.read_csv(test_file)
        
        # 2. Analisis OOV Karakter (Train vs Test)
        # Gunakan astype(str) untuk mencegah error jika ada data null/kosong
        train_text = "".join(df_train['sentence'].astype(str).tolist())
        test_text = "".join(df_test['sentence'].astype(str).tolist())
        
        train_chars = set(train_text)
        test_chars = set(test_text)
        
        # Cari karakter yang ada di Test tapi tidak ada di Train
        oov_chars = test_chars - train_chars
        
        test_text_len = len(test_text)
        if test_text_len > 0:
            oov_occurrences = sum(1 for char in test_text if char in oov_chars)
            oov_rate = (oov_occurrences / test_text_len) * 100
        else:
            oov_occurrences = 0
            oov_rate = 0.0

        # 3. Simpan metrik ke dalam list untuk tabel
        oov_summary_list.append({
            'Subject': subject,
            'Train (Baris)': len(df_train),
            'Val (Baris)': len(df_val),
            'Test (Baris)': len(df_test),
            'Kamus Train': len(train_chars),
            'Kamus Test': len(test_chars),
            'Karakter OOV': "".join(sorted(list(oov_chars))) if oov_chars else "-",
            'Total Muncul OOV': oov_occurrences,
            'OOV Rate (%)': round(oov_rate, 4)
        })
    else:
        # Jika salah satu dari 3 file per subjek tidak ada, berikan peringatan
        print(f"  ⚠️ Melewati {subject}: File train/val/test tidak lengkap di folder dataset.")

# ==========================================
# MENAMPILKAN TABEL SUMMARY OOV
# ==========================================
if oov_summary_list:
    df_summary = pd.DataFrame(oov_summary_list)

    # Menampilkan tabel (gunakan display agar format tabel interaktif Jupyter keluar)
    print("=" * 100)
    print("RINGKASAN DATA SPLIT & OOV KARAKTER PER SUBJEK")
    print("=" * 100)
    display(df_summary)
else:
    print("❌ Tidak ada data yang bisa ditampilkan. Pastikan file *_eq_3_0_*.csv sudah dibuat di dalam folder /dataset.")

Mengekstrak dan menghitung nilai CER dari seluruh eksperimen...

TABEL PERBANDINGAN PERFORMA CER EKSPERIMEN HILBERT


,Decoder,Train Data,SUB1,SUB2,SUB3,SUB4,SUB5,SUB6,SUB7,SUB8,SUB9,SUB10,SUB11,SUB12,Rata-rata Global
0,LSTM,Single,0.7318,0.7272,0.7242,0.7639,0.7615,0.7556,0.7665,0.7593,0.7529,0.7356,0.8070,0.8343,0.7600
1,IndoGPT,Single,0.7449,0.8009,0.9454,0.8043,0.8104,0.7844,0.7735,0.7408,0.7590,0.7486,0.7521,0.7391,0.7836
2,LSTM,All,0.7488,0.7347,0.7523,0.7488,0.7312,0.7412,0.7311,0.7342,0.7772,0.7363,0.7437,0.7950,0.7479
3,IndoGPT,All,0.7263,0.6979,0.7855,0.7758,0.7241,0.7606,0.7509,0.7236,0.7612,0.7220,0.7083,0.7360,0.7393


RINGKASAN DATA SPLIT & OOV KARAKTER PER SUBJEK


,Subject,Train (Baris),Val (Baris),Test (Baris),Kamus Train,Kamus Test,Karakter OOV,Total Muncul OOV,OOV Rate (%)
0,SUB1,70,10,20,24,23,-,0,0.0000
1,SUB2,35,5,10,23,23,z,1,0.3125
2,SUB3,35,5,10,23,21,-,0,0.0000
3,SUB4,70,10,20,24,22,-,0,0.0000
4,SUB5,35,5,10,23,23,-,0,0.0000
5,SUB6,70,10,20,23,22,-,0,0.0000
6,SUB7,70,10,20,24,22,-,0,0.0000
7,SUB8,70,10,20,23,23,-,0,0.0000
8,SUB9,64,9,19,24,22,-,0,0.0000
9,SUB10,63,9,19,23,23,-,0,0.0000


In [2]:
import pandas as pd
import numpy as np
import os
from IPython.display import display

# ==========================================
# BYPASS & IMPORT INDONLG TOKENIZER
# ==========================================
import transformers.utils
import transformers.utils.generic
import torch

# Bypass pengecekan TensorFlow dan tipe data untuk IndoBenchmark
if not hasattr(transformers.utils, 'is_tf_available'):
    transformers.utils.is_tf_available = lambda: False
if not hasattr(transformers.utils.generic, '_is_jax'):
    transformers.utils.generic._is_jax = lambda x: False
if not hasattr(transformers.utils.generic, '_is_tensorflow'):
    transformers.utils.generic._is_tensorflow = lambda x: False
if not hasattr(transformers.utils.generic, '_is_numpy'):
    transformers.utils.generic._is_numpy = lambda x: isinstance(x, np.ndarray)
if not hasattr(transformers.utils.generic, '_is_torch'):
    transformers.utils.generic._is_torch = lambda x: torch.is_tensor(x)
if not hasattr(transformers.utils.generic, '_is_torch_device'):
    transformers.utils.generic._is_torch_device = lambda x: isinstance(x, torch.device)

from indobenchmark import IndoNLGTokenizer

# ==========================================
# KONFIGURASI PATH & PARAMETER
# ==========================================
CURRENT_DIR = os.getcwd()
DATASET_DIR = os.path.abspath(os.path.join(CURRENT_DIR, '../../../dataset'))

subjects = [f'SUB{i}' for i in range(1, 13)]
oov_summary_list = []

print("Memuat IndoNLGTokenizer (IndoGPT)...")
tokenizer = IndoNLGTokenizer.from_pretrained("indobenchmark/indogpt")

# ==========================================
# PATCH: BYPASS ERROR 'padding_side'
# ==========================================
def dummy_pad(encoded_inputs, **kwargs):
    return encoded_inputs
tokenizer.pad = dummy_pad
# ==========================================

print(f"\nMembaca file split CSV dari direktori:\n{DATASET_DIR}\n")
print("Mulai menghitung metrik OOV Sub-word/Token untuk setiap subjek...\n")

for subject in subjects:
    train_file = os.path.join(DATASET_DIR, f"{subject}_eq_3_0_train.csv")
    val_file = os.path.join(DATASET_DIR, f"{subject}_eq_3_0_val.csv")
    test_file = os.path.join(DATASET_DIR, f"{subject}_eq_3_0_test.csv")
    
    if os.path.exists(train_file) and os.path.exists(val_file) and os.path.exists(test_file):
        # 1. Load Data
        df_train = pd.read_csv(train_file)
        df_val = pd.read_csv(val_file)
        df_test = pd.read_csv(test_file)
        
        # 2. Tokenisasi setiap kalimat menjadi daftar Token ID
        train_tokens = []
        for text in df_train['sentence'].astype(str):
            train_tokens.extend(tokenizer.encode(text))
            
        test_tokens = []
        for text in df_test['sentence'].astype(str):
            test_tokens.extend(tokenizer.encode(text))
        
        # 3. Cari Token (Sub-word) yang ada di Test tapi tidak di Train
        train_vocab = set(train_tokens)
        test_vocab = set(test_tokens)
        
        oov_token_ids = test_vocab - train_vocab
        
        # 4. Hitung statistik kemunculan OOV
        test_len = len(test_tokens)
        if test_len > 0:
            oov_occurrences = sum(1 for token in test_tokens if token in oov_token_ids)
            oov_rate = (oov_occurrences / test_len) * 100
        else:
            oov_occurrences = 0
            oov_rate = 0.0

        # 5. Decode token ID kembali menjadi teks untuk ditampilkan di tabel
        if oov_token_ids:
            # Decode masing-masing ID menjadi string sub-word
            oov_strings = [tokenizer.decode([t_id]).strip() for t_id in oov_token_ids]
            # Hapus string kosong jika ada, lalu gabungkan dengan koma
            oov_strings_clean = [s for s in oov_strings if s]
            oov_display = ", ".join(sorted(oov_strings_clean)) if oov_strings_clean else "-"
        else:
            oov_display = "-"

        # 6. Simpan metrik ke dalam list
        oov_summary_list.append({
            'Subject': subject,
            'Train (Baris)': len(df_train),
            'Val (Baris)': len(df_val),
            'Test (Baris)': len(df_test),
            'Kamus Train (Token)': len(train_vocab),
            'Kamus Test (Token)': len(test_vocab),
            'Sub-word OOV': oov_display,
            'Total Muncul OOV': oov_occurrences,
            'OOV Rate (%)': round(oov_rate, 4)
        })
    else:
        print(f"  ⚠️ Melewati {subject}: File train/val/test tidak lengkap di folder dataset.")

print("\n✅ Selesai memproses analisis OOV IndoGPT!\n")

# ==========================================
# MENAMPILKAN TABEL SUMMARY OOV
# ==========================================
if oov_summary_list:
    df_summary = pd.DataFrame(oov_summary_list)

    print("=" * 110)
    print("RINGKASAN OOV TOKEN (SUB-WORD INDOGPT) PER SUBJEK")
    print("=" * 110)
    display(df_summary)
else:
    print("❌ Tidak ada data yang bisa ditampilkan.")

Memuat IndoNLGTokenizer (IndoGPT)...



Membaca file split CSV dari direktori:
d:\GitRepos\EEG-to-Text_Conformer-Transducer\dataset

Mulai menghitung metrik OOV Sub-word/Token untuk setiap subjek...


✅ Selesai memproses analisis OOV IndoGPT!

RINGKASAN OOV TOKEN (SUB-WORD INDOGPT) PER SUBJEK


,Subject,Train (Baris),Val (Baris),Test (Baris),Kamus Train (Token),Kamus Test (Token),Sub-word OOV,Total Muncul OOV,OOV Rate (%)
0,SUB1,70,10,20,198,78,"baterainya, berjalan, buru, cepat, cobalah, da...",33,31.7308
1,SUB2,35,5,10,129,45,"berjalan, cepat, dari, kamar, kapan, lezat, ma...",24,47.0588
2,SUB3,35,5,10,127,38,"ada, adalah, akhirnya, benar, bicara, datang, ...",28,59.5745
3,SUB4,70,10,20,203,74,"akhir, bagaimana, baginya, bantuan, berharap, ...",41,45.0549
4,SUB5,35,5,10,121,44,"aku, banyak, bersenang, biasanya, diandalkan, ...",26,50.0000
5,SUB6,70,10,20,205,79,"alamat, atau, awal, ayam, bagaimana, bensin, b...",45,43.6893
6,SUB7,70,10,20,206,78,"apakah, bantuan, berkenalan, boleh, butuh, cob...",40,40.0000
7,SUB8,70,10,20,205,77,"ambilkan, berharap, bunga, buta, butuh, cepat,...",42,40.0000
8,SUB9,64,9,19,195,65,"bagaimana, baginya, berbicara, berkenalan, bic...",29,33.3333
9,SUB10,63,9,19,190,77,"aknya, banyak, berikan, besok, buru, dokumen, ...",37,38.9474
